# exp_002 · 트랙 A — 변이 유형 집계 피처

- **전역 실험 ID**: `iljun-logreg-002`  ·  **폴더**: `exp_002_variant_type` (GIT_STRATEGY §9.1)
- **Owner**: member_d (iljun) · **Seed**: 42
- **기준선**: `member-d-logreg-001` = Macro F1 **0.36305** (소급 변경 안 함 · §9.1)

## 이 폴더의 파일이 하는 일

| 파일 | 역할 | 실행 |
|---|---|---|
| `preprocessing/preprocess.py` | 팀 인터페이스 `fit/transform` (features_A 를 감쌈) | common 이 부름 |
| `training/model.py` `run.py` | 팀 방식 학습 (**holdout**) | `python -m ...training.run` |
| `pipeline.py` | 내 **CV·게이트·지문** 검증 | `python .../pipeline.py` |
| `experiment.ipynb` *(이 파일)* | 탐색·ablation | 매번 |

**두 진입점, 두 점수.** 팀 공식 결과는 `training.run`(holdout, `results/metrics.json`).
이 노트북과 `pipeline.py` 는 **CV** 로 탐색·검증하고 `results/metrics_cv.json` 을 남긴다.
GIT_STRATEGY §9.2 는 validation 객체에 StratifiedKFold 를 정식 허용하므로 CV 기록도 규약을 지킨다.

## 가설

`WT/변이` 이진화는 **어떤 유형의 변이인지**를 버린다. 클래스별 유형 구성이 실제로 다르다
(DLBC silent 41.20% vs LAML 17.68%; LAML frameshift 15.24% vs PAAD 0.76%).
→ 유형별 카운트를 넣으면 오르는가?

> ⏱ Run All 4\~6분.


## Section 1 · 설정 — 피처 본체와 파이프라인을 import

In [1]:
import sys, platform, time, importlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score


def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "configs" / "baseline.yaml").exists():
            return p
    raise FileNotFoundError("레포 루트를 못 찾음. 저장소 안에서 실행하세요.")


ROOT = find_root(Path.cwd())
EXP = ROOT / "experiments" / "member_d" / "exp_002_variant_type"
ART = EXP / "artifacts"; ART.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(EXP)); sys.path.insert(0, str(ROOT))

import features_A as fa
import pipeline as pa
importlib.reload(fa); importlib.reload(pa)
from features_A import KINDS, classify, parse_sample_counts, fit_spec, build_features

CFG = pa.load_cfg(); P = CFG["pipeline"]
REF = P["baseline"]; REF_F1, REF_ACC = REF["f1_macro"], REF["accuracy"]
DEC = P["decimals"]; SEED = P["cv"]["seed"]; N_SPLITS = P["cv"]["n_splits"]
TARGET, ID = "SUBCLASS", "ID"


def verdict(f1): return pa.verdict(f1, REF_F1, DEC)
def diff(f1): return f"{f1 - REF_F1:+.5f}"


print(f"experiment {CFG['experiment']['id']} · python {platform.python_version()}")
print(f"features_A {fa.__version__}  sha {pa.sha256(fa.__file__)[:12]}")
print(f"pipeline   {pa.PIPELINE_VERSION}  sha {pa.sha256(pa.__file__)[:12]}")
print(f"기준선 {REF['experiment']} F1 {REF_F1:.5f} → 판정값 {round(REF_F1, DEC)} · 게이트 {'ON' if P['submit_gate'] else 'OFF'}")

experiment iljun-logreg-002 · python 3.12.13
features_A A_v1_variant_type  sha d164d6e17c32
pipeline   exp002_v1  sha 2e8b50c94268
기준선 member-d-logreg-001 F1 0.36305 → 판정값 0.363 · 게이트 ON


### 데이터·파싱 — 파이프라인 함수를 그대로 쓴다

In [2]:
train, test, submission, gene_cols = pa.load_data(ROOT)
y_all = train[TARGET].values
cnt_train, cnt_test = pa.parse_all(train, test, gene_cols)

[Step 1] train (6201, 4386) · test (2546, 4385) · 유전자 4384
[Step 2] 파싱 3s


## Section 2 · 파서 확인

<callout>

**① 칸 내부 중복 토큰 1개로.** `"R248Q R248Q"` → 1건 (총 6,100건).
**② 판정 순서 고정.** `>` → `fs` → `del`/`ins` → 나머지. `^([A-Z*]+)(\d+)(.*)$`.
`TP469fs` 같은 두 글자 접두가 빠지지 않게 앞 아미노산을 여러 글자로 받는다.

</callout>

In [3]:
CASES = [("R248Q","missense"),("R248R","silent"),("R248*","nonsense"),
         ("TP469fs","frameshift"),("K57del","indel"),("468_469LG>F*","other"),("*261*","other")]
for tok, want in CASES:
    assert classify(tok) == want, f"{tok}: {classify(tok)} != {want}"
print(f"파서 단위 테스트 {len(CASES)}건 PASS\n")

tot = cnt_train[KINDS].sum().astype("int64")
print(pd.DataFrame({"건수": tot, "비중 %": (tot/tot.sum()*100).round(2)}).to_string())
print(f"\n합계 {int(tot.sum()):,}  (홍주님 표준안: missense 161,051 · silent 64,844)")

파서 단위 테스트 7건 PASS

                건수   비중 %
missense    161051  64.66
silent       64844  26.04
nonsense     13080   5.25
frameshift    9764   3.92
indel            3   0.00
other          322   0.13

합계 249,064  (홍주님 표준안: missense 161,051 · silent 64,844)


## Section 3 · 피처 블록

| 블록 | 내용 | 차원 |
|---|---|---|
| **G** | 유전자 이진화 (fold 상수열 제거) | ~4,230 |
| **B** | 변이 부담 log1p (유전자·이벤트·다중) | 3 |
| **V** | 유형 카운트 log1p | 6 |
| **R** | 유형 비율 | 6 |

모두 행 내부 연산이라 Leakage 아님. 배제한 것은 `features_A.py` docstring 참고.

In [4]:
spec = fit_spec(train, gene_cols, seed=SEED)
X, names = build_features(train, cnt_train, spec)
print(f"전체 블록 {X.shape} 밀도 {X.nnz/np.prod(X.shape)*100:.2f}% · 상수열 {len(gene_cols)}→{len(spec['keep_idx'])}")
print("피처 예시:", names[:2], "...", names[-3:])

전체 블록 (6201, 4245) 밀도 1.02% · 상수열 4384→4230
피처 예시: ['A_gene__A2M', 'A_gene__AAAS'] ... ['A_vratio__frameshift', 'A_vratio__indel', 'A_vratio__other']


## Section 4 · Leakage 자가검증 (파이프라인 Step 3 과 동일)

In [5]:
ok, _ = pa.leakage_checks(train, test, cnt_test, gene_cols, tuple(P["blocks"]), SEED)
assert ok

         [PASS] 부분집합 불변성 (앞 100행)
         [PASS] 단일 행 독립성
         [PASS] spec 재현성
         [PASS] NaN·inf 없음
         [PASS] test 결측 fillna 처리


## Section 5 · Ablation

`pa.cross_validate()` — fold 안에서 spec 재fit. 모델 고정, 피처 효과만 분리.

In [6]:
ABLATION = [("G","G        유전자만"),("GB","G+B      + 변이 부담"),
            ("GBV","G+B+V    + 유형 카운트"),("GBVR","G+B+V+R  + 유형 비율"),
            ("BVR","B+V+R    유전자 없이")]
results=[]
for b,l in ABLATION:
    r = pa.cross_validate(train, y_all, cnt_train, gene_cols, tuple(b),
                          model_key="logreg", seed=SEED, n_splits=N_SPLITS, label=l, v=False)
    r["label"]=l; results.append(r)
    print(f"{l:30} dim {r['dim']:5d}  F1 {r['f1_macro']:.5f} ({diff(r['f1_macro'])})  "
          f"Acc {r['accuracy']:.5f}  {verdict(r['f1_macro'])}")

G        유전자만                  dim  4226  F1 0.34469 (-0.01836)  Acc 0.34317  [-] 하락
G+B      + 변이 부담               dim  4229  F1 0.37429 (+0.01124)  Acc 0.36365  [+] 향상
G+B+V    + 유형 카운트              dim  4235  F1 0.38218 (+0.01913)  Acc 0.37558  [+] 향상
G+B+V+R  + 유형 비율               dim  4241  F1 0.38389 (+0.02084)  Acc 0.37671  [+] 향상
B+V+R    유전자 없이                dim    15  F1 0.15581 (-0.20724)  Acc 0.17336  [-] 하락


In [7]:
tab = pd.DataFrame([{k:r[k] for k in ("label","dim","f1_macro","accuracy")} for r in results])
tab["직전 대비"] = tab["f1_macro"].diff().round(5).where(tab.index<4)
tab["판정"] = [verdict(v) for v in tab["f1_macro"]]
print(tab.to_string(index=False))
tab.to_csv(ART/"ablation.csv", index=False, encoding="utf-8-sig")

            label  dim  f1_macro  accuracy   직전 대비     판정
    G        유전자만 4226   0.34469   0.34317     NaN [-] 하락
 G+B      + 변이 부담 4229   0.37429   0.36365 0.02960 [+] 향상
G+B+V    + 유형 카운트 4235   0.38218   0.37558 0.00789 [+] 향상
 G+B+V+R  + 유형 비율 4241   0.38389   0.37671 0.00171 [+] 향상
  B+V+R    유전자 없이   15   0.15581   0.17336     NaN [-] 하락


## Section 6 · 어느 클래스가 좋아졌나

In [8]:
classes = sorted(pd.unique(y_all))
best = max(results, key=lambda r: r["f1_macro"])
by = {r["blocks"]:r for r in results}
refr = by.get("GB") if by.get("GB") is not best else by.get("G", best)
d = pd.DataFrame({refr["blocks"]: f1_score(y_all, refr["oof"], average=None, labels=classes),
                  best["blocks"]+" (최고)": f1_score(y_all, best["oof"], average=None, labels=classes)},
                 index=classes)
d["변화"] = (d.iloc[:,1]-d.iloc[:,0]).round(4); d = d.round(4).sort_values("변화", ascending=False)
print(f"최고 {best['label'].strip()} ({best['f1_macro']:.5f}) · 비교 {refr['blocks']}\n")
print("오른 8개"); print(d.head(8).to_string())
print("\n내린 5개"); print(d.tail(5).to_string())
W=["LAML","DLBC","ACC","SKCM","THYM"]; w=d.loc[[c for c in W if c in d.index]]
print("\n가설 대상"); print(w.to_string())
print(f"\n{'[+]' if (w['변화']>0).sum()>=3 else '[-]'} 가설 대상 {len(w)}개 중 "
      f"{int((w['변화']>0).sum())}개 상승")
d.to_csv(ART/"class_f1_delta.csv", encoding="utf-8-sig")

최고 G+B+V+R  + 유형 비율 (0.38389) · 비교 GB

오른 8개
          GB  GBVR (최고)      변화
LUSC  0.2884     0.4037  0.1153
CESC  0.1221     0.1805  0.0584
PAAD  0.1942     0.2466  0.0524
STES  0.3476     0.3848  0.0372
SARC  0.1721     0.2057  0.0336
LUAD  0.2353     0.2655  0.0302
BRCA  0.4842     0.5104  0.0262
OV    0.3477     0.3633  0.0157

내린 5개
          GB  GBVR (최고)      변화
THYM  0.2881     0.2763 -0.0118
KIRC  0.1326     0.1152 -0.0174
ACC   0.8421     0.8182 -0.0239
DLBC  0.4643     0.4231 -0.0412
BLCA  0.3399     0.2857 -0.0542

가설 대상
          GB  GBVR (최고)      변화
LAML  0.5423     0.5382 -0.0040
DLBC  0.4643     0.4231 -0.0412
ACC   0.8421     0.8182 -0.0239
SKCM  0.7181     0.7302  0.0122
THYM  0.2881     0.2763 -0.0118

[-] 가설 대상 5개 중 1개 상승


## Section 7 · 모델 비교 (predict_proba 없는 LinearSVC 는 팀 프레임워크에서 못 씀)

In [9]:
bb = tuple(best["blocks"])
for k in ["logreg","svm","sgd"]:
    r = pa.cross_validate(train, y_all, cnt_train, gene_cols, bb,
                          model_key=k, seed=SEED, n_splits=N_SPLITS, v=False)
    print(f"{r['model_name']:32} F1 {r['f1_macro']:.5f} ({diff(r['f1_macro'])})  "
          f"Acc {r['accuracy']:.5f}  {verdict(r['f1_macro'])}")

LogisticRegression(balanced)     F1 0.38389 (+0.02084)  Acc 0.37671  [+] 향상
LinearSVC(balanced)              F1 0.30390 (-0.05915)  Acc 0.30560  [-] 하락
SGD(modified_huber, balanced)    F1 0.31715 (-0.04590)  Acc 0.32092  [-] 하락


## Section 8 · 파이프라인 교차검증 (검증 계약)

`pipeline.py` 를 CSV 부터 **처음부터** 돌려 노트북과 같은 CV 점수가 나오는지 assert 한다.
어긋나면 여기서 멈춘다.

In [10]:
champ = max(results, key=lambda r: r["f1_macro"])
res = pa.run_pipeline(root=ROOT, blocks=champ["blocks"], repeat=2)

same = res["f1_macro"] == champ["f1_macro"]
print("\n노트북 vs 파이프라인")
print(f"  Macro F1  {champ['f1_macro']:.5f} vs {res['f1_macro']:.5f}  [{'PASS' if same else 'FAIL'}]")
print(f"  결정성 (CV 2회 동일)                     [{'PASS' if res['deterministic'] else 'FAIL'}]")
assert same and res["deterministic"], "노트북↔파이프라인 또는 결정성 불일치"
print(f"\n{res['verdict']}  {res['experiment']}  F1 {res['f1_macro']:.5f} ({res['delta_vs_baseline']:+.5f})")
print(f"지문 pipeline {res['fingerprint']['pipeline_sha256'][:12]} · features {res['fingerprint']['features_sha256'][:12]}")

  iljun-logreg-002 · pipeline exp002_v1 · features_A A_v1_variant_type
  피처 GBVR · LogisticRegression(balanced) · seed 42 · StratifiedKFold-5
  기준선 member-d-logreg-001 = 0.36305 (판정 0.363)
[Step 1] train (6201, 4386) · test (2546, 4385) · 유전자 4384
[Step 2] 파싱 2s
         [PASS] 부분집합 불변성 (앞 100행)
         [PASS] 단일 행 독립성
         [PASS] spec 재현성
         [PASS] NaN·inf 없음
         [PASS] test 결측 fillna 처리
[Step 4] 교차검증 2회
         run 1/2                            dim  4241  F1 0.38389  Acc 0.37671  (20s)
         run 2/2                            dim  4241  F1 0.38389  Acc 0.37671  (20s)
         결정성 PASS
  Macro F1 0.38389 (+0.02084)  Acc 0.37671  [+] 향상
         [PASS] 행 수
         [PASS] 컬럼
         [PASS] 결측 없음
         [PASS] train 클래스 안
         [PASS] 쏠림 없음
[Step 6] results/metrics_cv.json (팀 metrics.json 은 training.run 몫)
  [+] 향상  iljun-logreg-002  F1 0.38389 (+0.02084)  Acc 0.37671
  지문 pipeline 2e8b50c94268 · features d164d6e17c32

노트북 vs 파이프라인
  Macro F1  0.38389 vs 0.383

### ✅ 실행 후 · 다음 단계

1. Section 5 `직전 대비` — 어느 블록이 일했나 (`B` 부담이 최대, `V` 그 위 추가, `R` 은 미미)
2. Section 6 가설 판정 — 오른 건 폐암 계열(LUSC 등)이지 유형 극단 클래스가 아님. **이게 트랙 B 실마리**
3. Section 8 두 줄 PASS — 노트북↔파이프라인 · 결정성

**팀 공식 결과(holdout)는 따로 남긴다:**
```bash
python -m experiments.member_d.exp_002_variant_type.training.run   # results/metrics.json (holdout)
python experiments/member_d/exp_002_variant_type/pipeline.py       # results/metrics_cv.json (CV·게이트·지문)
```

| 고칠 것 | 파일 |
|---|---|
| 피처 (핫스팟·truncating) | `features_A.py` → `build_features` 에 블록 H |
| 모델·파라미터 | `training/model.py` (팀) · `pipeline.py MODELS` (CV) |
| 기준선·게이트·블록 | `config.yaml` |
